In [4]:
!pip install google-search-results

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for google-search-results: filename=google_search_results-2.4.2-py3-none-any.whl size=32172 sha256=c2ae0e6ceea5d015eb53aec04eb115bc2f350f6dfac32693638438d168331cf8
  Stored in directory: c:\users\admin\appdata\local\pip\cache\wheels\44\af\e2\dde9fab6f1876485b72b35e9cd48da741da67d20e617c3b971
Successfully built google-search-results


In [12]:
import pandas as pd
from urllib.parse import urlparse
from serpapi import GoogleSearch
# Khởi tạo dataframe kết quả
df_cross = pd.DataFrame()

In [13]:
# ==============================
# TRUSTED DOMAINS & WEIGHTS
# ==============================

TRUSTED_DOMAINS = {
    # Quốc tế - rất cao
    "reuters.com": 1.0,
    "bbc.com": 1.0,
    "apnews.com": 1.0,
    "who.int": 1.0,
    "nytimes.com": 1.0,
    # Thông tấn & Tài chính quốc tế
    "afp.com": 1.0,          # Agence France-Presse
    "bloomberg.com": 1.0,    # Bloomberg
    "forbes.com": 1.0,       # Forbes
    "wsj.com": 1.0,          # The Wall Street Journal
    "theguardian.com": 0.95, # The Guardian
    "aljazeera.com": 0.95,   # Al Jazeera

    # Việt Nam - cao
    "vnexpress.net": 0.9,
    "tuoitre.vn": 0.9,
    "thanhnien.vn": 0.9,
    "vietnamnet.vn": 0.9,
    "dantri.com.vn": 0.9,
    "vna.vn": 1.0,           # Thông tấn xã Việt Nam
    "vietnamplus.vn": 1.0,   # Báo điện tử đối ngoại quốc gia
    "vov.vn": 1.0,           # Đài Tiếng nói Việt Nam
    "vtv.vn": 1.0,           # Đài Truyền hình Việt Nam
    "nhandan.vn": 1.0,       # Báo Nhân Dân
    "chinhphu.vn": 1.0,      # Cổng Thông tin điện tử Chính phủ
    "baochinhphu.vn": 1.0,   # Báo Chính phủ
    "laodong.vn": 0.9,       # Báo Lao Động
    "sggp.org.vn": 0.9,      # Sài Gòn Giải Phóng
    "qdnd.vn": 0.9,          # Báo Quân đội nhân dân
    "cand.com.vn": 0.9,      # Báo Công an nhân dân

    # Y tế Quốc tế & Việt Nam
    "cdc.gov": 1.0,          # CDC Hoa Kỳ
    "mayoclinic.org": 1.0,   # Mayo Clinic
    "moh.gov.vn": 1.0,       # Bộ Y tế Việt Nam
    "hcdc.vn": 0.95,         # Trung tâm Kiểm soát Bệnh tật TP.HCM
    "nihe.org.vn": 1.0,      # Viện Vệ sinh Dịch tễ Trung ương

    # Fact-checking
    "fullfact.org": 1.0,
    "snopes.com": 1.0,
    "politifact.com": 1.0,
    "factcheck.org": 1.0,
    "vncrc.gov.vn": 1.0,     # Trung tâm Xử lý tin giả Việt Nam

    # Mạng xã hội / video – thấp
    "facebook.com": 0.2,
    "youtube.com": 0.2,
    "tiktok.com": 0.1
}

def extract_domain(url):
    try:
        return urlparse(url).netloc.replace("www.", "")
    except:
        return ""


In [14]:
query = "COVID-19 có thể chữa khỏi trong 3 ngày"

In [ ]:
SERPAPI_KEY = "enter your API key here"
params = {
    "engine": "google",
    "q": query,
    "hl": "vi",
    "gl": "vn",
    "num": 10,
    "api_key": SERPAPI_KEY
}

search = GoogleSearch(params)
results = search.get_dict()

In [16]:
organic_results = results.get("organic_results", [])

all_hits = []
trusted_hits = []

for r in organic_results:
    title = r.get("title", "")
    link = r.get("link", "")
    snippet = r.get("snippet", "")

    domain = extract_domain(link)

    hit = {
        "title": title,
        "url": link,
        "snippet": snippet,
        "domain": domain,
        "is_trusted": domain in TRUSTED_DOMAINS,
        "weight": TRUSTED_DOMAINS.get(domain, 0.0)
    }

    all_hits.append(hit)

    if hit["is_trusted"]:
        trusted_hits.append(hit)
print(f"🔎 Tổng số kết quả: {len(all_hits)}")
print(f"✅ Trusted sources: {len(trusted_hits)}")

🔎 Tổng số kết quả: 9
✅ Trusted sources: 2


In [17]:
# ==============================
# CROSS SOURCE SCORING
# ==============================

if len(trusted_hits) == 0:
    cross_score = 0.2
elif len(trusted_hits) == 1:
    cross_score = 0.6
else:
    cross_score = min(
        1.0,
        sum(h["weight"] for h in trusted_hits) / 2
    )

print("🔎 Trusted sources found:", len(trusted_hits))
print("📊 Cross-source score:", cross_score)

🔎 Trusted sources found: 2
📊 Cross-source score: 0.95


In [18]:
rows = []

for h in all_hits:
    rows.append({
        "query": query,
        "result_title": h["title"],
        "result_url": h["url"],
        "snippet": h["snippet"],
        "domain": h["domain"],
        "is_trusted": h["is_trusted"],
        "domain_weight": h["weight"],
        "cross_score": cross_score
    })

df_temp = pd.DataFrame(rows)
df_cross = pd.concat([df_cross, df_temp], ignore_index=True)
df_cross.head()

,query,result_title,result_url,snippet,domain,is_trusted,domain_weight,cross_score
0,COVID-19 có thể chữa khỏi trong 3 ngày,Điều trị covid-19 bao lâu thì khỏi? 3 giai đoạ...,https://hellobacsi.com/benh-truyen-nhiem/coron...,Thời gian điều trị và phục hồi sau khi nhiễm C...,hellobacsi.com,False,0.0,0.95
1,COVID-19 có thể chữa khỏi trong 3 ngày,Điều trị COVID-19 bao lâu thì khỏi? 4 giai đoạ...,https://www.nhathuocankhang.com/ban-tin-suc-kh...,sẽ biến mất hoàn toàn là khoảng 2 tuần nếu bện...,nhathuocankhang.com,False,0.0,0.95
2,COVID-19 có thể chữa khỏi trong 3 ngày,Bị Covid mấy ngày thì khỏi? Những điều cần biế...,https://nhathuoclongchau.com.vn/bai-viet/bi-co...,Bệnh nhân bị Covid có thể khỏi trong 1 – 2 tuầ...,nhathuoclongchau.com.vn,False,0.0,0.95
3,COVID-19 có thể chữa khỏi trong 3 ngày,Thời gian bị Covid-19 bao lâu thì khỏi?,https://tiemchunglongchau.com.vn/kien-thuc-tie...,"Với những trường hợp mắc bệnh nhẹ, không có bi...",tiemchunglongchau.com.vn,False,0.0,0.95
4,COVID-19 có thể chữa khỏi trong 3 ngày,Một số trường hợp tái dương tính sau khi đã kh...,https://moh.gov.vn/tin-lien-quan/-/asset_publi...,"Tuy nhiên, trong thực tế có không ít trường hợ...",moh.gov.vn,True,1.0,0.95


In [19]:
import os
os.makedirs("ML/output", exist_ok=True)
df_cross.to_csv(
    "ML/output/cross_check_results.csv",
    index=False,
    encoding="utf-8-sig"
)

print("✅ Cross-check results saved")

✅ Cross-check results saved
